# MCAM Keyword Generator — Colab Launcher

This notebook launches the **MCAM (Museum Collection Art Metadata) keyword generator**, a tool that suggests relevant art-history keywords for museum collection images. It clones the project repo, loads ~44K AAT (Art & Architecture Thesaurus) terms into a vector database, and starts a Gradio web UI where you can upload images and receive keyword suggestions.

## Configuration

These constants control how the notebook behaves. The most common ones to tweak:
- `TERM_COUNT` — how many keywords the UI returns per image
- `CANDIDATE_POOL_SIZE` — how many candidates to fetch before MMR diversity selection and reranking
- `MMR_LAMBDA` — balance between relevance (1.0) and diversity (0.0); default 0.7
- `MIN_SIMILARITY_THRESHOLD` — discard candidates below this cosine similarity
- `USE_FLASH_ATTENTION` — set to `True` for faster inference on L4/A100 GPUs
- `FORCE_REBUILD_COLLECTION` — set to `True` if you changed the AAT dataset and need to re-embed

In [ ]:
import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cs2535-oakhoury-2026spring/Mills-Museum.git"
REPO_BRANCH = "restructure-v1"
REPO_DIR = Path("/content/Mills-Museum")
DATA_DIR = REPO_DIR / "data"
EXPORT_DIR = REPO_DIR / "exports"
EMBED_MODEL_REPO = "Qwen/Qwen3-VL-Embedding-2B"
RERANK_MODEL_REPO = "Qwen/Qwen3-VL-Reranker-2B"
COLLECTION_NAME = "aat_terms"
VDB_PATH = REPO_DIR / ".colab_chroma"
TERM_COUNT = 10
CANDIDATE_POOL_SIZE = 80
AAT_BATCH_SIZE = 24
USE_FLASH_ATTENTION = False
FORCE_REBUILD_COLLECTION = False
MMR_LAMBDA = 0.7
MIN_SIMILARITY_THRESHOLD = 0.15

def run(cmd, cwd=None, check=True):
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    subprocess.run(cmd, cwd=cwd, check=check)

## Clone or Update the Repository

This pulls the project source code from GitHub. On first run it clones the repo; on subsequent runs it fetches and fast-forward merges the latest changes from the configured branch.

In [ ]:
if not REPO_DIR.exists():
    run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

## Install Dependencies

Upgrades pip and installs all Python packages listed in `requirements.txt`, plus Gradio (for the web UI), the HuggingFace Hub client, and Accelerate (for efficient model loading).

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    "requirements.txt",
    "gradio",
    "huggingface_hub",
    "hf_xet",
    "accelerate",
])

## Device Detection and Flash Attention

Detects whether a CUDA GPU is available and selects the appropriate device. If `USE_FLASH_ATTENTION` is enabled and a GPU is present, it installs and activates Flash Attention 2 for significantly faster transformer inference. The notebook works fine on CPU too, just slower.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

attn_kwargs = {}
if USE_FLASH_ATTENTION and device == "cuda":
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "ninja", "flash-attn", "--no-build-isolation"])
        attn_kwargs["attn_implementation"] = "flash_attention_2"
        print("Flash Attention enabled.")
    except Exception as exc:
        print(f"Flash Attention install failed, continuing without it: {exc}")

## Download Qwen Model Helper Scripts

The Qwen3-VL embedding and reranking models ship their own Python helper scripts inside their HuggingFace repos. This section downloads those `scripts/` folders and merges them into a local `vendor/qwen_scripts` directory so they can be imported as regular Python modules.

In [ ]:
from huggingface_hub import snapshot_download

vendor_root = REPO_DIR / "vendor" / "qwen_scripts"
scripts_root = vendor_root / "scripts"
scripts_root.mkdir(parents=True, exist_ok=True)

def merge_repo_scripts(repo_id: str) -> None:
    snapshot_dir = Path(snapshot_download(repo_id=repo_id, allow_patterns=["scripts/**"]))
    source_root = snapshot_dir / "scripts"
    if not source_root.exists():
        raise FileNotFoundError(f"Repo {repo_id} did not expose a scripts/ folder.")
    for path in source_root.rglob("*"):
        if not path.is_file():
            continue
        target_path = scripts_root / path.relative_to(source_root)
        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target_path)

merge_repo_scripts(EMBED_MODEL_REPO)
merge_repo_scripts(RERANK_MODEL_REPO)

if str(vendor_root) not in sys.path:
    sys.path.insert(0, str(vendor_root))

## Load AAT Terms and Set Up ChromaDB

This section defines helpers to load the AAT (Art & Architecture Thesaurus) dataset (~44K standardized art terms) from the repo's `data/` folder. It supports parquet, CSV, TSV, and Excel formats with flexible column-name detection. It also initializes a persistent ChromaDB vector store that will hold the embedded term vectors for similarity search.

In [ ]:
import chromadb
import pandas as pd
from datasets import Dataset
from tqdm.auto import tqdm

from scripts.qwen3_vl_embedding import Qwen3VLEmbedder
from scripts.qwen3_vl_reranker import Qwen3VLReranker

def load_repo_aat_terms() -> Dataset:
    candidate_patterns = [
        "**/*aat*.parquet",
        "**/*keyword*.parquet",
        "**/*aat*.csv",
        "**/*keyword*.csv",
        "**/*aat*.tsv",
        "**/*keyword*.tsv",
        "**/*aat*.xlsx",
        "**/*keyword*.xlsx",
    ]

    column_aliases = {
        "term_label": ["term_label", "keyword", "label", "preferred_term"],
        "term_note": ["term_note", "scope_note", "note", "description"],
        "term_id": ["term_id", "aat_id", "id"],
    }

    matched_files = []
    for pattern in candidate_patterns:
        matched_files.extend(sorted(DATA_DIR.glob(pattern)))

    if not matched_files:
        raise FileNotFoundError(
            f"No AAT term file was found under {DATA_DIR}. Put a parquet, csv, tsv, or xlsx file there."
        )

    for file_path in matched_files:
        suffix = file_path.suffix.lower()
        print(f"Loading repo AAT terms from {file_path}")
        if suffix == ".parquet":
            frame = pd.read_parquet(file_path)
        elif suffix == ".csv":
            frame = pd.read_csv(file_path)
        elif suffix == ".tsv":
            frame = pd.read_csv(file_path, sep="\t")
        elif suffix == ".xlsx":
            frame = pd.read_excel(file_path)
        else:
            continue

        normalized = pd.DataFrame()
        for target_name, aliases in column_aliases.items():
            source_name = next((alias for alias in aliases if alias in frame.columns), None)
            if source_name is not None:
                normalized[target_name] = frame[source_name]

        if "term_label" not in normalized.columns:
            print(f"Skipping {file_path}: could not find a term label column.")
            continue

        if "term_note" not in normalized.columns:
            normalized["term_note"] = None
        if "term_id" not in normalized.columns:
            normalized["term_id"] = normalized.index.astype(str)

        normalized = normalized[["term_label", "term_note", "term_id"]].fillna("")
        normalized["term_label"] = normalized["term_label"].astype(str)
        normalized["term_note"] = normalized["term_note"].astype(str)
        normalized["term_id"] = normalized["term_id"].astype(str)
        return Dataset.from_pandas(normalized, preserve_index=False)

    raise ValueError(
        f"Found candidate files in {DATA_DIR}, but none exposed a usable term label column."
    )

def find_repo_images() -> list[str]:
    image_suffixes = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".webp"}
    image_paths = []
    for path in sorted(DATA_DIR.glob("**/*")):
        if path.suffix.lower() in image_suffixes:
            image_paths.append(str(path.resolve()))
    return image_paths

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
VDB_PATH.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(VDB_PATH))
collection = None

if FORCE_REBUILD_COLLECTION:
    try:
        chroma_client.delete_collection(name=COLLECTION_NAME)
        print(f"Deleted existing collection '{COLLECTION_NAME}' because FORCE_REBUILD_COLLECTION=True.")
    except Exception:
        pass

try:
    collection = chroma_client.get_collection(name=COLLECTION_NAME)
    print(f"Reusing existing Chroma collection '{COLLECTION_NAME}' with {collection.count()} items.")
except Exception:
    print(f"Creating Chroma collection '{COLLECTION_NAME}'.")
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

## Embedding Model and AAT Indexing Loop

Loads the Qwen3-VL embedding model and uses it to convert every AAT term into a vector, storing them in the ChromaDB collection. This is the slowest step on first run (it processes ~44K terms in batches) but is fully cached -- subsequent runs skip it if the collection already exists.

In [ ]:
embedding_model = Qwen3VLEmbedder(
    model_name_or_path=EMBED_MODEL_REPO,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    **attn_kwargs,
)

if collection.count() == 0:
    aat_terms = load_repo_aat_terms()
    for i in tqdm(range(0, len(aat_terms), AAT_BATCH_SIZE), desc="Embedding AAT terms"):
        batch = aat_terms.select(range(i, min(i + AAT_BATCH_SIZE, len(aat_terms))))
        batch_ids = [str(j) for j in range(i, i + len(batch))]
        batch_doc_texts = [f"{row['term_label']} : {row.get('term_note') or ''}" for row in batch]
        batch_metadata = [
            {"term_label": row["term_label"], "term_id": str(row.get("term_id", ""))}
            for row in batch
        ]
        qwen_inputs = [{"text": text} for text in batch_doc_texts]
        embeddings = embedding_model.process(qwen_inputs)
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        collection.add(
            ids=batch_ids,
            embeddings=embeddings.cpu().float().tolist(),
            documents=batch_doc_texts,
            metadatas=batch_metadata,
        )
    print(f"Finished building '{COLLECTION_NAME}' with {collection.count()} AAT terms.")
else:
    print("Skipping AAT embedding because the local Chroma collection is already populated.")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Load Reranking Model

Loads the Qwen3-VL reranker, a second model that takes the top candidate keywords from the vector search and reorders them by fine-grained relevance to the input image. This two-stage retrieve-then-rerank pipeline produces more accurate keyword suggestions than embedding search alone.

In [ ]:
reranking_model = Qwen3VLReranker(
    model_name_or_path=RERANK_MODEL_REPO,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    **attn_kwargs,
)

repo_image_paths = find_repo_images()
print(f"Found {len(repo_image_paths)} image file(s) under {DATA_DIR}.")

## Launch the Gradio Web UI

Configures and launches the Gradio-based web interface. The app is served with `share=True`, which generates a public URL so you can access the UI from any browser. Upload a museum collection image and the pipeline will suggest relevant AAT keywords ranked by relevance.

In [ ]:
import src.frontend.gradio as app

app.DEFAULT_KEYWORD_COUNT = TERM_COUNT
app.DEFAULT_CANDIDATE_POOL_SIZE = CANDIDATE_POOL_SIZE
app.MMR_LAMBDA = MMR_LAMBDA
app.MIN_SIMILARITY_THRESHOLD = MIN_SIMILARITY_THRESHOLD
app.configure_backend_runtime(
    collection=collection,
    embedding_model=embedding_model,
    reranking_model=reranking_model,
)
app.configure_demo_assets(
    image_paths=repo_image_paths,
    export_dir=str(EXPORT_DIR),
)

allowed_paths = [str(DATA_DIR.resolve()), str(EXPORT_DIR.resolve())]
print("Launching Gradio interface...")
app.interface.launch(share=True, allowed_paths=allowed_paths)